In [11]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "dblp-1.csv").exists():
    ROOT_DIR = ROOT_DIR / "project"
DATA_DIR = ROOT_DIR / "data"

In [12]:
con = duckdb.connect(database=":memory:")

dblp_1_path = DATA_DIR / "dblp-1.csv"
dblp_2_path = DATA_DIR / "dblp-2.csv"
dblp_3_path = DATA_DIR / "dblp-3.csv"
dblp_4_path = DATA_DIR / "dblp-4.csv"

df = con.query(f"""
    SELECT * FROM read_csv_auto('{dblp_1_path}')
    UNION ALL
    SELECT * FROM read_csv_auto('{dblp_2_path}')
    UNION ALL
    SELECT * FROM read_csv_auto('{dblp_3_path}')
    UNION ALL
    SELECT * FROM read_csv_auto('{dblp_4_path}')
""")

df.show()

┌──────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬──────────┬────────────┬─────────┬─────────┬────────────────────────────────┬──────────┬─────────────┬───────────────┬─────────────────┬───────────────────┬───────────┐
│ column00 │                                                                   pauthor                                                                   │ peditor │                                                 ptitle                                                 │ pyear │ paddress │ ppublisher │ pseries │   pid   │              pkey              │ ptype_id │ pjournal_id │ pbooktitle_id │ pjournalfull_id │ pbooktitlefull_id │ partition │
│  int64   │                                                                   varchar                      

### Using pandas for the json files

In [13]:
def load_lookup(path):
    with open(path) as f:
        raw = json.load(f)
    df_lu = pd.DataFrame(raw) 
    return dict(zip(df_lu["id"], df_lu["name"]))

In [14]:
ptype        = load_lookup(DATA_DIR / "ptype.json")
pjournal     = load_lookup(DATA_DIR / "pjournal.json")
pjournalfull = load_lookup(DATA_DIR / "pjournalfull.json")
pbooktitle   = load_lookup(DATA_DIR / "pbooktitle.json")
pbooktitlefull = load_lookup(DATA_DIR / "pbooktitlefull.json")

id_mappings = {
    "ptype_id":          ptype,
    "pjournal_id":       pjournal,
    "pbooktitle_id":     pbooktitle,
    "pjournalfull_id":   pjournalfull,
    "pbooktitlefull_id": pbooktitlefull,
}

for name, m in [("ptype", ptype), ("pjournal", pjournal),
                ("pjournalfull", pjournalfull), ("pbooktitle", pbooktitle),
                ("pbooktitlefull", pbooktitlefull)]:
    print(f"{name}: {len(m)} rows")

ptype: 9 rows
pjournal: 718 rows
pjournalfull: 90 rows
pbooktitle: 2720 rows
pbooktitlefull: 1084 rows


Transforming the dicts from the json files into duckdb tables in order to perform JOIN

In [15]:
ptype_df = pd.DataFrame(list(ptype.items()), columns=["id", "name"])
con.register("ptype_df", ptype_df)

pjournal_df = pd.DataFrame(list(pjournal.items()), columns=["id", "name"])
pjournalfull_df = pd.DataFrame(list(pjournalfull.items()), columns=["id", "name"])
con.register("pjournal_df", pjournal_df)
con.register("pjournalfull_df", pjournalfull_df)

pbooktitle_df = pd.DataFrame(list(pbooktitle.items()), columns=["id", "name"])
pbooktitlefull_df = pd.DataFrame(list(pbooktitlefull.items()), columns=["id", "name"])
con.register("pbooktitle_df", pbooktitle_df)
con.register("pbooktitlefull_df", pbooktitlefull_df)

In [16]:
pbooktitle_df

,id,name
0,0,nán
1,1,VLSI Design
2,2,Prozeßrechnersysteme
3,3,PRICAI
4,4,DFT
...,...,...
2715,2715,Symposium on Programming
2716,2716,TLDI
2717,2717,ECML Workshop on Learning Contex-Free Grammars
2718,2718,"Concurrency, Graphs and Models"


## Missing/NULL/NaN values

In [17]:
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df
    WHERE pbooktitle_id = 0
""").show()

┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│       1686 │
└────────────┘



## Dropping NULL Columns
- paddress
- ppublisher
- pseries
- peditor

In [18]:
con.query("SELECT COUNT(*) FROM df").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        17165 │
└──────────────┘



In [19]:
df_new = con.query("""
        SELECT 
            df.pauthor, 
            df.ptitle, 
            df.pyear, 
            df.pid, 
            df.pkey,
            df.ptype_id,
            df.pjournal_id,
            df.pjournalfull_id,
            df.pbooktitle_id,
            df.pbooktitlefull_id,
            ptype_df.name         AS ptype,
            pjournal_df.name      AS journal,
            pjournalfull_df.name  AS journalfull,
            pbooktitle_df.name    AS booktitle,
            pbooktitlefull_df.name AS booktitlefull
        FROM df 
        LEFT JOIN ptype_df
            ON df.ptype_id = ptype_df.id
        LEFT JOIN pjournal_df
            ON df.pjournal_id = pjournal_df.id
        LEFT JOIN pjournalfull_df
            ON df.pjournalfull_id = pjournalfull_df.id
        LEFT JOIN pbooktitle_df
            ON df.pbooktitle_id = pbooktitle_df.id
        LEFT JOIN pbooktitlefull_df
            ON df.pbooktitlefull_id = pbooktitlefull_df.id
    """)

In [20]:
con.query("SELECT COUNT(*) FROM df_new").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        17165 │
└──────────────┘



In [21]:
df_new.show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬─────────┬────────────────────────────────┬──────────┬─────────────┬─────────────────┬───────────────┬───────────────────┬───────────────┬─────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────────────────┐
│                                                                  pauthor                                                                  │                                                    ptitle                                                     │ pyear │   pid   │              pkey              │ ptype_id │ pjournal_id │ pjournalfull_id │ pbooktitle_id │ pbooktitlefull_

In [22]:
con.query("""
    SELECT 'pauthor' AS column_name, COUNT(*) AS null_count FROM df_new WHERE pauthor IS NULL OR pauthor = '' OR pauthor = 'nan'
    UNION ALL
    SELECT 'ptitle',                  COUNT(*) FROM df_new WHERE ptitle IS NULL OR ptitle = '' OR ptitle = 'nan'
    UNION ALL
    SELECT 'pyear',                   COUNT(*) FROM df_new WHERE pyear IS NULL
    UNION ALL
    SELECT 'pid',                     COUNT(*) FROM df_new WHERE pid IS NULL
    UNION ALL
    SELECT 'pkey',                    COUNT(*) FROM df_new WHERE pkey IS NULL
    UNION ALL
    SELECT 'ptype',                   COUNT(*) FROM df_new WHERE ptype IS NULL
""").show()

┌─────────────┬────────────┐
│ column_name │ null_count │
│   varchar   │   int64    │
├─────────────┼────────────┤
│ pauthor     │          0 │
│ ptitle      │          0 │
│ pyear       │          0 │
│ pid         │          0 │
│ pkey        │          0 │
│ ptype       │          0 │
└─────────────┴────────────┘



## Booktitle

In [23]:
# booktitle and booktitle full check for NULL, nan and Messung values
print("Checking booktitle for NULL, 'nán', and 'Messung' values:")
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df_new
    WHERE booktitle IS NULL OR booktitle = 'nán' OR booktitle = 'Messung'
    """).show()

print("Checking booktitlefull for NULL, 'nán', and 'Messung' values:")
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df_new
    WHERE booktitlefull IS NULL OR booktitlefull = 'nán' OR booktitlefull = 'Messung'
    """).show()

Checking booktitle for NULL, 'nán', and 'Messung' values:
┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│       8040 │
└────────────┘

Checking booktitlefull for NULL, 'nán', and 'Messung' values:
┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│      11287 │
└────────────┘



Booktitlefull has 7 more missing values than booktitle, so lets check where those occur:

In [24]:
con.query("""
    SELECT booktitle, booktitlefull, COUNT(*) AS cnt
    FROM df_new
    WHERE (booktitlefull IS NULL OR booktitlefull = 'nán' OR booktitlefull = 'Messung')
      AND NOT (booktitle IS NULL OR booktitle = 'nán' OR booktitle = 'Messung')
    GROUP BY booktitle, booktitlefull
    ORDER BY cnt DESC
""").show()

┌───────────────────────────────────────────────┬───────────────┬───────┐
│                   booktitle                   │ booktitlefull │  cnt  │
│                    varchar                    │    varchar    │ int64 │
├───────────────────────────────────────────────┼───────────────┼───────┤
│ CVPR                                          │ NULL          │    19 │
│ STOC                                          │ NULL          │    12 │
│ MICAI                                         │ NULL          │    10 │
│ CIARP                                         │ NULL          │    10 │
│ ACM Great Lakes Symposium on VLSI             │ NULL          │     9 │
│ ICIAP                                         │ NULL          │     9 │
│ CADE                                          │ NULL          │     9 │
│ DSN                                           │ NULL          │     9 │
│ Advances in Computer Entertainment Technology │ NULL          │     9 │
│ ISMIR                               

In [25]:
con.query("""
    SELECT booktitle, booktitlefull
    FROM df_new
    WHERE booktitle = 'DFT'
""").show()

┌───────────┬───────────────┐
│ booktitle │ booktitlefull │
│  varchar  │    varchar    │
├───────────┼───────────────┤
│ DFT       │ nán           │
│ DFT       │ nán           │
│ DFT       │ nán           │
│ DFT       │ nán           │
│ DFT       │ nán           │
│ DFT       │ nán           │
│ DFT       │ nán           │
└───────────┴───────────────┘



So whenever booktitle has DFT theres a nan value in booktitle full. One approach is to make those nan values in booktitlefull also equal to DFT

In [26]:
# duckdb can't update a view result directly, so we will do the update in pandas and then re-register the DataFrame
df_new_pandas = df_new.to_df()
mask = (df_new_pandas['booktitlefull'] == 'nán') & (df_new_pandas['booktitle'] == 'DFT')
df_new_pandas.loc[mask, 'booktitlefull'] = 'DFT'
con.register("df_new", df_new_pandas)

print("Checking AGAIN booktitlefull for NULL, 'nán', and 'Messung' values:")
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df_new
    WHERE booktitlefull IS NULL OR booktitlefull = 'nán' OR booktitlefull = 'Messung'
    """).show()

Checking AGAIN booktitlefull for NULL, 'nán', and 'Messung' values:
┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│      11280 │
└────────────┘



Lastly we can check if the rows in which booktitle has a missing value are the same as when booktitlefull has missing values

In [27]:
con.query("""
    SELECT COUNT(*) AS only_booktitle_missing
    FROM df_new
    WHERE (booktitle IS NULL OR booktitle = 'nán' OR booktitle = 'Messung')
      AND NOT (booktitlefull IS NULL OR booktitlefull = 'nán' OR booktitlefull = 'Messung')
""").show()

┌────────────────────────┐
│ only_booktitle_missing │
│         int64          │
├────────────────────────┤
│                      0 │
└────────────────────────┘



They are the same. We will make them NULL later

## Journal
Repeating the same process above but for pjournal and pjournalfull

In [28]:
print("Checking journal for NULL, 'nán', 'nan', 'NaN', and 'Messung' values:")
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df_new
    WHERE journal IS NULL OR journal = 'nán' OR journal = 'nan' OR journal = 'NaN' OR journal = 'Messung'
    """).show()

print("Checking journalfull for NULL, 'nán', 'nan', 'NaN', and 'Messung' values:")
con.query("""
    SELECT COUNT(*) AS null_count
    FROM df_new
    WHERE journalfull IS NULL OR journalfull = 'nán' OR journalfull = 'nan' OR journalfull = 'NaN' OR journalfull = 'Messung'
    """).show()

Checking journal for NULL, 'nán', 'nan', 'NaN', and 'Messung' values:
┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│      11610 │
└────────────┘

Checking journalfull for NULL, 'nán', 'nan', 'NaN', and 'Messung' values:
┌────────────┐
│ null_count │
│   int64    │
├────────────┤
│      15128 │
└────────────┘



They have the same number of values, lets check if the entries match

In [29]:
con.query("""
    SELECT COUNT(*) AS only_journal_missing
    FROM df_new
    WHERE (journal IS NULL OR journal = 'nán' OR journal = 'nan' OR journal = 'NaN' OR journal = 'Messung')
      AND NOT (journalfull IS NULL OR journalfull = 'nán' OR journalfull = 'nan' OR journalfull = 'NaN' OR journalfull = 'Messung')
""").show()

┌──────────────────────┐
│ only_journal_missing │
│        int64         │
├──────────────────────┤
│                    0 │
└──────────────────────┘



Okay here they also occur in the same rows. Lets also make these all NULL

In [30]:
missing_vals = ['nán', 'nan', 'NaN', 'Messung']

df_new_pandas['booktitle'] = df_new_pandas['booktitle'].replace(missing_vals, None)
df_new_pandas['booktitlefull'] = df_new_pandas['booktitlefull'].replace(missing_vals, None)
df_new_pandas['journal'] = df_new_pandas['journal'].replace(missing_vals, None)
df_new_pandas['journalfull'] = df_new_pandas['journalfull'].replace(missing_vals, None)

con.register("df_new", df_new_pandas)

In [31]:
df_new_pandas.head()

,pauthor,ptitle,pyear,pid,pkey,ptype_id,pjournal_id,pjournalfull_id,pbooktitle_id,pbooktitlefull_id,ptype,journal,journalfull,booktitle,booktitlefull
0,Jorge Semião|Juan J. Rodríguez-Andina|Fabian V...,Improving the Tolerance of Pipeline Based Circ...,-2007,180843,conf/dft/SemiaoRVSTT07,1,0,0,4,4,inproceedings,NaN,NaN,DFT,DFT
1,Patrice Caire,A Normative Multi-Agent Systems Approach to th...,-2007,162991,conf/dagstuhl/Caire07,2,0,0,7,7,inprớcéédings,NaN,NaN,Normative Multi-agent Systems,Éúrớpéán Grid Cớnféréncé
2,Sundeep B|Andrew Thangaraj,Self-Orthogonality of q-Ary Images of qm-Ary C...,2007,2261406,journals/tit/BT07,0,2,2,9,9,árticlé,IEEE Transactions on Information Theory,International Journal of Ambient Computing and...,WiMớb,ACM Symposium on Parallel Algorithms and Archi...
3,Gerardo Pardo-Castellote,OMG Data-Distribution Service: Architectural O...,-2003,349720,conf/icdcsw/Pardo-Castellote03,1,0,0,11,11,inproceedings,NaN,NaN,ICDCS Workshops,International Agent Technology Conference
4,Ki-Hoon Lee|Kyu-Young Whang|Wook-Shin Han|Min-...,Structural Consistency: Enabling XML Keyword S...,2009,1922328,journals/corr/abs-0911-4329,3,5,5,5,5,article,CoRR,International Journal of Wireless Information ...,NaN,None


In [32]:
con.query("""
    SELECT 'booktitle'     AS column_name, COUNT(*) AS null_count FROM df_new WHERE booktitle IS NULL
    UNION ALL
    SELECT 'booktitlefull' AS column_name, COUNT(*) AS null_count FROM df_new WHERE booktitlefull IS NULL
    UNION ALL
    SELECT 'journal'       AS column_name, COUNT(*) AS null_count FROM df_new WHERE journal IS NULL
    UNION ALL
    SELECT 'journalfull'   AS column_name, COUNT(*) AS null_count FROM df_new WHERE journalfull IS NULL
""").show()

┌───────────────┬────────────┐
│  column_name  │ null_count │
│    varchar    │   int64    │
├───────────────┼────────────┤
│ booktitle     │       8040 │
│ booktitlefull │      11280 │
│ journal       │      11610 │
│ journalfull   │      15128 │
└───────────────┴────────────┘



In [33]:
con.query("SELECT * FROM df_new").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬─────────┬────────────────────────────────┬──────────┬─────────────┬─────────────────┬───────────────┬───────────────────┬───────────────┬─────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────────────────┐
│                                                                  pauthor                                                                  │                                                    ptitle                                                     │ pyear │   pid   │              pkey              │ ptype_id │ pjournal_id │ pjournalfull_id │ pbooktitle_id │ pbooktitlefull_

In [34]:
df_new_pandas.head(10)

,pauthor,ptitle,pyear,pid,pkey,ptype_id,pjournal_id,pjournalfull_id,pbooktitle_id,pbooktitlefull_id,ptype,journal,journalfull,booktitle,booktitlefull
0,Jorge Semião|Juan J. Rodríguez-Andina|Fabian V...,Improving the Tolerance of Pipeline Based Circ...,-2007,180843,conf/dft/SemiaoRVSTT07,1,0,0,4,4,inproceedings,NaN,NaN,DFT,DFT
1,Patrice Caire,A Normative Multi-Agent Systems Approach to th...,-2007,162991,conf/dagstuhl/Caire07,2,0,0,7,7,inprớcéédings,NaN,NaN,Normative Multi-agent Systems,Éúrớpéán Grid Cớnféréncé
2,Sundeep B|Andrew Thangaraj,Self-Orthogonality of q-Ary Images of qm-Ary C...,2007,2261406,journals/tit/BT07,0,2,2,9,9,árticlé,IEEE Transactions on Information Theory,International Journal of Ambient Computing and...,WiMớb,ACM Symposium on Parallel Algorithms and Archi...
3,Gerardo Pardo-Castellote,OMG Data-Distribution Service: Architectural O...,-2003,349720,conf/icdcsw/Pardo-Castellote03,1,0,0,11,11,inproceedings,NaN,NaN,ICDCS Workshops,International Agent Technology Conference
4,Ki-Hoon Lee|Kyu-Young Whang|Wook-Shin Han|Min-...,Structural Consistency: Enabling XML Keyword S...,2009,1922328,journals/corr/abs-0911-4329,3,5,5,5,5,article,CoRR,International Journal of Wireless Information ...,NaN,None
5,Bin Liu|David J. Hill|Kok Lay Teo,Input-to-state stability for a class of hybrid...,-2009,883711,conf/cdc/LiuHT09,1,0,0,5,5,inproceedings,NaN,NaN,NaN,None
6,A fourth-order compact difference scheme on fa...,Haiwei Sun|Ning Kang|Jun Zhang|Eric S. Carlson,-2003,2324621,journals/mcs/SunKZC03,3,6,6,5,5,article,Mathematics and Computers in Simulation,International Transactions on Systems Science ...,NaN,None
7,Sang-Heon Lee|Nam-Sung Kim|Heau-Jo Kang|Soon-G...,Performance Improvement of Intelligent UWB-IR ...,-2006,366780,conf/icic/LeeKKK06,0,0,0,0,0,árticlé,NaN,NaN,None,NaN
8,Eugene Pamba Capo-Chichi|Hervé Guyennet|Jean-M...,IEEE 802.15.4 Performance on a Hierarchical Hy...,-2009,794722,conf/icns/Capo-ChichiGF09,2,0,0,15,15,inprớcéédings,NaN,NaN,ICNS,Ásiá-Pácific Wéb Cớnféréncé
9,Feng Wang|Michael J. Hannafin,Scaffolding preservice teachers' WebQuest desi...,-2009,2339884,journals/jche/WangH09,0,10,10,5,5,árticlé,J. Computing in Higher Education,Journal of Computers,NaN,None


#### Cleaning Strings of Booktitles

In [35]:
import unicodedata
import re

In [36]:
def clean_string(s):
    if s is None or pd.isnull(s):
        return s

    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))

    s = s.replace("-", " ")
    return s.strip().lower()

In [37]:
for id, name in df_new_pandas['booktitlefull'].items():
    #print(df_new_pandas.loc[id, 'booktitlefull'])
    df_new_pandas.loc[id, 'booktitlefull'] = clean_string(name)

for id, name in df_new_pandas['booktitle'].items():
    df_new_pandas.loc[id, 'booktitle'] = clean_string(name)

df_new_pandas.head()

,pauthor,ptitle,pyear,pid,pkey,ptype_id,pjournal_id,pjournalfull_id,pbooktitle_id,pbooktitlefull_id,ptype,journal,journalfull,booktitle,booktitlefull
0,Jorge Semião|Juan J. Rodríguez-Andina|Fabian V...,Improving the Tolerance of Pipeline Based Circ...,-2007,180843,conf/dft/SemiaoRVSTT07,1,0,0,4,4,inproceedings,NaN,NaN,dft,dft
1,Patrice Caire,A Normative Multi-Agent Systems Approach to th...,-2007,162991,conf/dagstuhl/Caire07,2,0,0,7,7,inprớcéédings,NaN,NaN,normative multi agent systems,european grid conference
2,Sundeep B|Andrew Thangaraj,Self-Orthogonality of q-Ary Images of qm-Ary C...,2007,2261406,journals/tit/BT07,0,2,2,9,9,árticlé,IEEE Transactions on Information Theory,International Journal of Ambient Computing and...,wimob,acm symposium on parallel algorithms and archi...
3,Gerardo Pardo-Castellote,OMG Data-Distribution Service: Architectural O...,-2003,349720,conf/icdcsw/Pardo-Castellote03,1,0,0,11,11,inproceedings,NaN,NaN,icdcs workshops,international agent technology conference
4,Ki-Hoon Lee|Kyu-Young Whang|Wook-Shin Han|Min-...,Structural Consistency: Enabling XML Keyword S...,2009,1922328,journals/corr/abs-0911-4329,3,5,5,5,5,article,CoRR,International Journal of Wireless Information ...,NaN,None


In [38]:
def extract_abbreviations(name):
    if not name or pd.isnull(name):
        return None

    # Extracting ABBREV in 'Full Title (ABBREV)'
    match = re.search(r'\(([^)]+)\)$', name.strip())
    if match:
        return match.group(1).strip()
   
   # Extracting ABBREV in'Full Title - ABBREV'
    match = re.search(r'\s[-–]\s([A-Z][^\s].*)$', name.strip())
    if match:
        return match.group(1).strip()
    
    return None

In [39]:
journal_abbreviations = {k: extract_abbreviations(name) for k, name in pjournalfull.items()}

In [40]:
def clean_journal_titles(name):

    if not name or pd.isnull(name):
        return name
    
    # remove abbreviations and acronyms 
    match = re.search(r'\(([^)]+)\)$', name.strip())
    if match:
        name = name.replace(match.group(0), "")
    
    match = re.search(r'\s[-–]\s([A-Z][^\s].*)$', name.strip())
    if match:
        name = name.replace(match.group(0), "")

    # then cleaning string normally
    cleaned = clean_string(name)
    return cleaned

In [41]:
for id, name in df_new_pandas['journal'].items():
    #print(df_new_pandas.loc[id, 'booktitlefull'])
    df_new_pandas.loc[id, 'journal'] = clean_string(name)

for id, name in df_new_pandas['journalfull'].items():
    df_new_pandas.loc[id, 'journalfull'] = clean_string(name)

for id, abb in journal_abbreviations.items():
    df_new_pandas.loc[id, 'journal_abbreviation'] = clean_string(abb)

In [42]:
df_new_pandas.head()

,pauthor,ptitle,pyear,pid,pkey,ptype_id,pjournal_id,pjournalfull_id,pbooktitle_id,pbooktitlefull_id,ptype,journal,journalfull,booktitle,booktitlefull,journal_abbreviation
0,Jorge Semião|Juan J. Rodríguez-Andina|Fabian V...,Improving the Tolerance of Pipeline Based Circ...,-2007,180843,conf/dft/SemiaoRVSTT07,1,0,0,4,4,inproceedings,NaN,NaN,dft,dft,None
1,Patrice Caire,A Normative Multi-Agent Systems Approach to th...,-2007,162991,conf/dagstuhl/Caire07,2,0,0,7,7,inprớcéédings,NaN,NaN,normative multi agent systems,european grid conference,None
2,Sundeep B|Andrew Thangaraj,Self-Orthogonality of q-Ary Images of qm-Ary C...,2007,2261406,journals/tit/BT07,0,2,2,9,9,árticlé,ieee transactions on information theory,international journal of ambient computing and...,wimob,acm symposium on parallel algorithms and archi...,ijaci
3,Gerardo Pardo-Castellote,OMG Data-Distribution Service: Architectural O...,-2003,349720,conf/icdcsw/Pardo-Castellote03,1,0,0,11,11,inproceedings,NaN,NaN,icdcs workshops,international agent technology conference,monet
4,Ki-Hoon Lee|Kyu-Young Whang|Wook-Shin Han|Min-...,Structural Consistency: Enabling XML Keyword S...,2009,1922328,journals/corr/abs-0911-4329,3,5,5,5,5,article,corr,international journal of wireless information ...,NaN,None,None


In [43]:
con.register("df_new", df_new_pandas)

## New DuckDB Table

Currently this table was still registered to a pandas dataframe. So I'm creating a new duckdb table to make the next changes on that directly. 

In [44]:
con.query("""
    CREATE OR REPLACE TABLE ducktable AS
    SELECT * FROM df_new
""")

## PYear 
Taking the absolute value of the negative years

In [45]:
con.query("""
    UPDATE ducktable
    SET pyear = ABS(pyear)
    WHERE pyear < 0
""")

## Publication Type: ptype_id

Changing the typeids using a CASE statement: 
remap corrupted IDs to canonical ones: `{0→3, 2→1, 5→6, 7→8}`

In [46]:
con.query(""" SELECT COUNT(*) FROM ducktable""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        17165 │
└──────────────┘



In [47]:
# first I will drop the existing column of ptype
con.query("""ALTER TABLE ducktable DROP COLUMN ptype""")

# then I will alter the values of the ids to the correct ones
con.query("""
    UPDATE ducktable
    SET ptype_id = CASE ptype_id
        WHEN 0 THEN 3
        WHEN 2 THEN 1
        WHEN 5 THEN 6
        WHEN 7 THEN 8
        ELSE ptype_id
    END
""")

# then I will join with the ptype_df to get the correct ptype names
con.query("""
    CREATE OR REPLACE TABLE ducktable AS
    SELECT ducktable.*, ptype_df.name AS ptype
    FROM ducktable
    LEFT JOIN ptype_df ON ducktable.ptype_id = ptype_df.id
""")

In [48]:
con.query(""" SELECT COUNT(*) FROM ducktable""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        17165 │
└──────────────┘



In [50]:
con.query(""" SELECT * FROM ducktable""").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬─────────┬────────────────────────────────┬──────────┬─────────────┬─────────────────┬───────────────┬───────────────────┬─────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────────────────┬──────────────────────┬───────────────┐
│                                                                  pauthor                                                                  │                                                    ptitle                                                     │ pyear │   pid   │              pkey              │ ptype_id │ pjournal_id │ pjournalfull_id │ pbooktit

## PKey

## Swapping Author and Title
- Performing the swap when ptitle has a "|"

In [51]:
con.query("""
    UPDATE ducktable
    SET 
        ptitle  = pauthor,
        pauthor = ptitle
    WHERE ptitle LIKE '%|%'
""")

Now we take care of the rows that have swapped author and title when there is only one author (that the previous cell might have missed)

In [52]:
con.query("""
    UPDATE ducktable
    SET
        ptitle  = pauthor,
        pauthor = ptitle
    WHERE
        LENGTH(ptitle) < LENGTH(pauthor) AND 
        (LOWER(pauthor) LIKE '% of %' OR LOWER(pauthor) LIKE '% the %' 
        OR LOWER(pauthor) LIKE '% for %' OR LOWER(pauthor) LIKE '% on %'
        OR LOWER(pauthor) LIKE '% in %' OR LOWER(pauthor) LIKE '% with %' OR LOWER(pauthor) LIKE '% is %') 
    """)

con.query(""" SELECT ptitle, pauthor FROM ducktable""").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                  ptitle                                                                   │                                                        pauthor                                                        │
│                                                                  varchar                                                                  │                                                        varchar                                                        │
├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────────────

We can also perform the swap when authors end with a period - that should be a title

In [53]:
con.query("""
    UPDATE ducktable
    SET ptitle = pauthor, pauthor = ptitle
    WHERE pauthor LIKE '%.'
    AND ptitle NOT LIKE '%.'
""")

## Normalise Author and Title

In [54]:
con.query(r"""
    UPDATE ducktable
    SET 
        ptitle  = LOWER(TRIM(REGEXP_REPLACE(ptitle, '\s+', ' ', 'g'))),
        pauthor = LOWER(TRIM(REGEXP_REPLACE(pauthor, '\s+', ' ', 'g')))
""")

## Saving Clean Data

In [55]:
con.query("""SELECT * FROM ducktable""").show()

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┬─────────┬────────────────────────────────┬──────────┬─────────────┬─────────────────┬───────────────┬───────────────────┬─────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────────────────┬──────────────────────┬───────────────┐
│                                                        pauthor                                                        │                                                                  ptitle                                                                   │ pyear │   pid   │              pkey              │ ptype_id │ pjournal_id │ pjournalfu

In [56]:
final_pandas_df = con.query("""SELECT * FROM ducktable""").to_df()
final_pandas_df.drop(columns=['ptype_id', 'pjournal_id', 'pjournalfull_id', 'pbooktitle_id', 'pbooktitlefull_id'], inplace=True)
final_pandas_df.head()

,pauthor,ptitle,pyear,pid,pkey,journal,journalfull,booktitle,booktitlefull,journal_abbreviation,ptype
0,jorge semião|juan j. rodríguez-andina|fabian v...,improving the tolerance of pipeline based circ...,2007,180843,conf/dft/SemiaoRVSTT07,NaN,NaN,dft,dft,NaN,inproceedings
1,patrice caire,a normative multi-agent systems approach to th...,2007,162991,conf/dagstuhl/Caire07,NaN,NaN,normative multi agent systems,european grid conference,NaN,inproceedings
2,sundeep b|andrew thangaraj,self-orthogonality of q-ary images of qm-ary c...,2007,2261406,journals/tit/BT07,ieee transactions on information theory,international journal of ambient computing and...,wimob,acm symposium on parallel algorithms and archi...,ijaci,article
3,gerardo pardo-castellote,omg data-distribution service: architectural o...,2003,349720,conf/icdcsw/Pardo-Castellote03,NaN,NaN,icdcs workshops,international agent technology conference,monet,inproceedings
4,ki-hoon lee|kyu-young whang|wook-shin han|min-...,structural consistency: enabling xml keyword s...,2009,1922328,journals/corr/abs-0911-4329,corr,international journal of wireless information ...,NaN,NaN,NaN,article


In [57]:
final_pandas_df.to_csv(DATA_DIR / "duckdb_cleaned_dblp_v2.csv", index=False)